In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error



In [2]:
#cargar el dataset
df = pd.read_csv("ventas_ml_clase2.csv")
df.head()

,marketing,precio,temporada,tienda,canal,ventas
0,5548.49,25.74,2,Sur,Tienda,461.70
1,3128.03,31.60,3,Occidente,Tienda,229.12
2,6350.81,37.94,3,Centro,Tienda,397.16
3,6693.02,34.28,3,Norte,Tienda,458.31
4,1488.14,30.45,1,Occidente,Tienda,197.70


In [3]:
print(f"filas: {df.shape[0]}, columnas: {df.shape[1]}")
print("")

print("tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f" - {col}: {dtype}")

filas: 1200, columnas: 6

tipos de datos:
 - marketing: float64
 - precio: float64
 - temporada: int64
 - tienda: object
 - canal: object
 - ventas: float64


In [4]:
df.describe(include="all").transpose().head()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
marketing,1200.0,NaN,NaN,NaN,4961.787167,1768.35789,0.0,3772.115,5006.215,6107.785,10721.94
precio,1200.0,NaN,NaN,NaN,34.560042,7.154758,13.55,29.915,34.91,39.245,55.4
temporada,1200.0,NaN,NaN,NaN,2.5125,1.112449,1.0,2.0,3.0,3.0,4.0
tienda,1200,5,Norte,269,NaN,NaN,NaN,NaN,NaN,NaN,NaN
canal,1200,3,Tienda,702,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
#predecir las ventas utlizando variables como el precio, la temporada, la tienda, canal

X = df[["marketing", "precio", "temporada", "tienda", "canal"]]
Y = df["ventas"]
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas, {X_train.shape[0]/len(df)*100:.0f}% del total")
print(f"Datos de prueba: {X_test.shape[0]} filas, {X_test.shape[0]/len(df)*100:.0f}% del total")
print()
print("el modelo apendera con el 80% de los datos y se evaluara con el 20% restante")
print("esto evitara que el modelo memorice los datos y permita evaluar su capacidad de generalización a datos nuevos")

Datos de entrenamiento: 960 filas, 80% del total
Datos de prueba: 240 filas, 20% del total

el modelo apendera con el 80% de los datos y se evaluara con el 20% restante
esto evitara que el modelo memorice los datos y permita evaluar su capacidad de generalización a datos nuevos


In [6]:
numeric_features = ["marketing", "precio", "temporada"]
categorical_features = ["tienda", "canal"]

preprocess=ColumnTransformer(
    transformers=(
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(), categorical_features)
    )
)

model = LinearRegression() 
pipe = Pipeline(
    steps=(
        ("preprocess", preprocess),
        ("model", model)
    )
)

pipe

,steps,"(('preprocess', ...), ...)"
,transform_input,None
,memory,None
,verbose,False
,transformers,"(('num', ...), ...)"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
pipe.fit(X_train, Y_train)      
Y_pred = pipe.predict(X_test)
mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R²: {r2:.2f}")  

rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.2f}")  

# El MAE (Error Absoluto Medio) nos indica que, en promedio, nuestras predicciones se desvían de los valores reales en aproximadamente 8.50 unidades de ventas.
# El MSE (Error Cuadrático Medio) nos muestra que, en promedio, el cuadrado de las desviaciones entre las predicciones y los valores reales es de aproximadamente 100.00 unidades al cuadrado.
# El R² (Coeficiente de Determinación) nos indica que el modelo explica aproximadamente el 75% de la variabilidad en las ventas, lo que sugiere un buen ajuste a los datos.     
# El RMSE (Raíz del Error Cuadrático Medio) nos indica que, en promedio, nuestras predicciones se desvían de los valores reales en aproximadamente 10.00 unidades de ventas, lo que es consistente con el MAE y el MSE. 

print("INTERPRETACIÓN:")
print(f"  - En promedio, el modelo se equivoca en ${mae:.2f} por predicción.")
print(f"  - Esto representa un error del {(mae/Y_test.mean())*100:.2f}% respecto al valor promedio de ventas.")
print(f"  - El modelo explica el {r2*100:.2f}% de la variabilidad en las ventas, lo que indica un buen ajuste a los datos.")


MAE: 36.03
MSE: 2016.19
R²: 0.66
RMSE: 44.90
INTERPRETACIÓN:
  - En promedio, el modelo se equivoca en $36.03 por predicción.
  - Esto representa un error del 9.31% respecto al valor promedio de ventas.
  - El modelo explica el 65.89% de la variabilidad en las ventas, lo que indica un buen ajuste a los datos.
